In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os, zipfile, random, shutil
from collections import defaultdict

random.seed(42)
PATCHES_PER_PATIENT = 2000

working_dir = "/kaggle/working"
for item in os.listdir(working_dir):
    item_path = os.path.join(working_dir, item)
    if item == ".virtual_documents":
        continue
    if os.path.isdir(item_path):
        shutil.rmtree(item_path)
    else:
        os.remove(item_path)
print("Working directory cleared")

import urllib.request

# ── E2 ONLY ──────────────────────────────────────────────
url_e2 = "https://zenodo.org/records/5337009/files/TCGA-BRCA-E2-DEEPMED-TILES.zip?download=1"
zip_e2 = "/kaggle/working/E2_full.zip"
extract_e2 = "/kaggle/working/tiles/E2/extracted_full"

os.makedirs(extract_e2, exist_ok=True)

print("Downloading E2 (10.3GB)... keep tab active or use commit")
urllib.request.urlretrieve(url_e2, zip_e2)
size_gb = os.path.getsize(zip_e2) / (1024**3)
print(f"Downloaded: {size_gb:.2f} GB")

print("Verifying integrity...")
with zipfile.ZipFile(zip_e2, 'r') as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f"Corrupted: {bad}")
print("Integrity check PASSED")

print("Extracting 2000 patches per patient (E2)...")
with zipfile.ZipFile(zip_e2, 'r') as zf:
    jpg_names = [n for n in zf.namelist() if n.endswith(".jpg")]
    patches_by_patient = defaultdict(list)
    for name in jpg_names:
        patient_id = name.split("/")[1][:12]
        patches_by_patient[patient_id].append(name)
    print(f"Unique patients found: {len(patches_by_patient)}")
    extracted = 0
    for patient_id, patch_list in patches_by_patient.items():
        selected = (patch_list if len(patch_list) <= PATCHES_PER_PATIENT
                    else random.sample(patch_list, PATCHES_PER_PATIENT))
        for patch_name in selected:
            zf.extract(patch_name, extract_e2)
            extracted += 1
    print(f"Total patches extracted: {extracted}")

os.remove(zip_e2)
print("E2 zip deleted")

print("\nDisk usage:")
os.system("df -h /kaggle/working")
print("\nE2 extraction complete.")

Working directory cleared
Downloaded: 10.31 GB
Verifying integrity...
Integrity check PASSED
Extracting 2000 patches per patient (E2)...
Unique patients found: 90
Total patches extracted: 154150
E2 zip deleted

Disk usage:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.9G   12G  41% /kaggle/working

E2 extraction complete.
